## 导入相关库

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from keras.callbacks import ModelCheckpoint, EarlyStopping

2026-08-29 15:40:15.967444: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-29 15:40:16.015679: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-29 15:40:16.015714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-29 15:40:16.017153: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-29 15:40:16.025296: I tensorflow/core/platform/cpu_feature_guar

## 模型结构

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.regularizers import l2


class AdaptiveFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma, alpha, delta, reduction=tf.keras.losses.Reduction.AUTO, name='AdaptiveFocalLoss'):
        super().__init__(reduction=reduction, name=name)
        self.gamma = gamma
        self.alpha = alpha
        self.delta = delta
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        p_t = tf.clip_by_value(p_t, self.delta, 1.0 - self.delta)
        alpha_factor = tf.where(tf.equal(y_true, 1), self.alpha, 1 - self.alpha)
        focal_loss = -alpha_factor * tf.pow(1 - p_t, self.gamma) * tf.math.log(p_t)
        return tf.reduce_mean(focal_loss)
    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma, "alpha": self.alpha, "delta": self.delta})
        return config

class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', threshold=0.5, **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        tp = tf.reduce_sum(y_true * y_pred)
        fp = tf.reduce_sum(y_pred) - tp
        fn = tf.reduce_sum(y_true) - tp
        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)
    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + 1e-6)
        recall = self.true_positives / (self.true_positives + self.false_negatives + 1e-6)
        return 2 * (precision * recall) / (precision + recall + 1e-6)
    def reset_states(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)




#===========================
# 多尺度 ConvNeXtBlock1D
# ============================
class ConvNeXtBlock1D(layers.Layer):
    def __init__(self, filters, kernel_sizes=[3, 5, 7], drop_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_sizes = kernel_sizes
        self.drop_rate = drop_rate
        self.conv_dw_branches = []
        for ks in kernel_sizes:
            self.conv_dw_branches.append(
                layers.Conv1D(filters, ks, padding='same', kernel_regularizer=l2(1e-4))
            )
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.mlp_dense1 = layers.Dense(filters * 4, activation='gelu', kernel_regularizer=l2(1e-4))
        self.mlp_dense2 = layers.Dense(filters, kernel_regularizer=l2(1e-4))
        self.dropout = layers.Dropout(drop_rate)
        self.proj = None

    def build(self, input_shape):
        in_channels = input_shape[-1]
        if in_channels != self.filters:
            self.proj = layers.Conv1D(self.filters, kernel_size=1, padding='same', kernel_regularizer=l2(1e-4))
        super().build(input_shape)

    def call(self, x, training=None):
        residual = x
        multi_out = self.conv_dw_branches[0](x)
        for branch in self.conv_dw_branches[1:]:
            multi_out = multi_out + branch(x)
        x = multi_out
        x = self.norm(x)
        x = self.mlp_dense1(x)
        x = self.mlp_dense2(x)
        x = self.dropout(x, training=training)
        if self.proj is not None:
            residual = self.proj(residual)
        return tf.nn.relu(x + residual)

# ======================== 模型构建 ========================
def build_optimized_model(input_shape, lr):
    inputs = layers.Input(shape=input_shape)
    x = ConvNeXtBlock1D(256, kernel_sizes=[3, 5, 7], drop_rate=0.1)(inputs)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)
    x = ConvNeXtBlock1D(128, kernel_sizes=[3, 5, 7], drop_rate=0.1)(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Bidirectional(
        layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                   kernel_regularizer=l2(1e-4), recurrent_regularizer=l2(1e-4))
    )(x)
    x = layers.Bidirectional(
        layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                   kernel_regularizer=l2(1e-4), recurrent_regularizer=l2(1e-4))
    )(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, kernel_regularizer=l2(1e-4))(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, kernel_regularizer=l2(1e-4))(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=AdaptiveFocalLoss(gamma=2.0, alpha=0.5, delta=0.02),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.Precision(name='precision'),
                 F1Score()]
    )
    return model

## 用到的操作函数

In [3]:
# ======================== 数据加载与采样 ========================
def process_files_to_arrays(filenames):
    X_all_chr = []
    middle_row_indices_all_chr = []
    y_all_chr = []
    sample_counts = []
    for filename in filenames:
        count = 0
        with open(filename, 'r') as file:
            lines = file.readlines()
            for line in lines:
                line = line.strip()
                if line:
                    data = eval(line, {"array": np.array})
                    X_all_chr.append(data[0])
                    middle_row_indices_all_chr.append(data[1])
                    y_all_chr.append(data[2])
                    count += 1
        sample_counts.append(count)
    return (np.array(X_all_chr), np.array(middle_row_indices_all_chr),
            np.array(y_all_chr), sample_counts)

def process_and_merge_chromosomes(X, mid_indices, y, sample_counts, select):
    X_selected_list = []
    mid_selected_list = []
    y_selected_list = []
    start_idx = 0
    for chr_idx, count in enumerate(sample_counts):
        end_idx = start_idx + count
        X_chr = X[start_idx:end_idx]
        mid_chr = mid_indices[start_idx:end_idx]
        y_chr = y[start_idx:end_idx]
        X_sel, mid_sel, y_sel = select(X_chr, mid_chr, y_chr, 5)
        X_selected_list.append(X_sel)
        mid_selected_list.append(mid_sel)
        y_selected_list.append(y_sel)
        pos = np.sum(y_sel == 1)
        neg = np.sum(y_sel == 0)
        print(f"Chromosome {chr_idx+1} processed | pos: {pos} | neg: {neg} | total: {len(y_sel)}")
        start_idx = end_idx
    X_selected = np.concatenate(X_selected_list, axis=0)
    mid_selected = np.concatenate(mid_selected_list, axis=0)
    y_selected = np.concatenate(y_selected_list, axis=0)
    print(f"\nAll merged | feature shape: {X_selected.shape} | pos ratio: {np.sum(y_selected==1)}/{len(y_selected)}")
    return X_selected, mid_selected, y_selected

def print_class_distribution(y, description=""):
    unique, counts = np.unique(y, return_counts=True)
    class_distribution = dict(zip(unique, counts))
    print(f"{description} sample distribution:")
    print(f"Positive (1): {class_distribution.get(1, 0)}")
    print(f"Negative (0): {class_distribution.get(0, 0)}")
    print(f"Total: {sum(counts)}")
    print(f"Ratio: {class_distribution.get(1, 0)/sum(counts):.2%} positive\n")

def select(X_train, middle_row_indices_train, y_train, safe_radius):
    positive_mask = (y_train == 1)
    negative_mask = ~positive_mask
    positive_indices = np.where(positive_mask)[0]
    negative_indices = np.where(negative_mask)[0]
    if len(positive_indices) == 0:
        return X_train, middle_row_indices_train, y_train
    positive_positions = middle_row_indices_train[positive_indices]
    negative_positions = middle_row_indices_train[negative_indices]
    distances = np.abs(negative_positions[:, np.newaxis] - positive_positions)
    min_distances = np.min(distances, axis=1)
    safe_mask = min_distances > safe_radius
    safe_negative_indices = negative_indices[safe_mask]
    n_pos = len(positive_indices)
    n_neg_desired = n_pos
    if len(safe_negative_indices) >= n_neg_desired:
        selected_neg = np.random.choice(safe_negative_indices, n_neg_desired, replace=False)
    else:
        selected_neg = np.random.choice(safe_negative_indices, n_neg_desired, replace=True)
    selected_indices = np.concatenate([positive_indices, selected_neg])
    np.random.shuffle(selected_indices)
    return (X_train[selected_indices], middle_row_indices_train[selected_indices], y_train[selected_indices])

# ======================== 转移概率计算 ========================
def compute_node_degrees(H):
    return np.sum(H, axis=1)
def compute_hyperedge_degrees(H):
    return np.sum(H, axis=0)
def compute_first_order_transition_probabilities(H, node_degrees, hyperedge_degrees):
    num_nodes, num_hyperedges = H.shape
    P1 = np.zeros((num_nodes, num_nodes))
    for v in range(num_nodes):
        for u in range(num_nodes):
            if u == v:
                continue
            pi_uv = 0
            for e in range(num_hyperedges):
                if H[u, e] == 0 or H[v, e] == 0:
                    continue
                h_ve = H[v, e]
                h_ue = H[u, e]
                d_v = node_degrees[v]
                delta_e = hyperedge_degrees[e]
                pi_uv += (h_ve * h_ue) / (d_v * delta_e)
            P1[v, u] = pi_uv
    return P1
def generate_P(data):
    P = []
    for H in data:
        node_degrees = compute_node_degrees(H)
        hyperedge_degrees = compute_hyperedge_degrees(H)
        P1 = compute_first_order_transition_probabilities(H, node_degrees, hyperedge_degrees)
        P.append(P1)
    return np.array(P)

# ======================== 实验核心函数 ========================
def run_experiment(seed, X_selected_train, y_selected_train,
                   X_selected_val, y_selected_val, X_test_filenames):
    X_train, y_train_bal = shuffle(X_selected_train, y_selected_train, random_state=seed)
    X_val, y_val_bal = shuffle(X_selected_val, y_selected_val, random_state=seed)
    P_train = generate_P(X_train)
    P_val = generate_P(X_val)
    input_shape = (11, 11)
    all_lr_results = []
    lrs = [0.001, 0.0001, 0.003, 0.0003]
    for lr in lrs:
        tf.random.set_seed(seed)
        model = build_optimized_model(input_shape, lr)
        class_weights = compute_class_weight('balanced', classes=np.unique(y_train_bal), y=y_train_bal)
        class_weight_dict = dict(enumerate(class_weights))
        checkpoint_dir = f'checkpoints/seed_{seed}/lr_{lr}'
        os.makedirs(checkpoint_dir, exist_ok=True)
        checkpoint_path = os.path.join(checkpoint_dir, f'best_model_seed_{seed}_lr_{lr}.weights.h5')
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        callbacks = [
            ModelCheckpoint(filepath=checkpoint_path, monitor='val_f1_score',
                            save_best_only=True, save_weights_only=True, mode='max', verbose=1),
            EarlyStopping(monitor='val_f1_score', mode='max', patience=30,
                          restore_best_weights=True, verbose=1)
        ]
        history = model.fit(P_train, y_train_bal,
                            validation_data=(P_val, y_val_bal),
                            batch_size=32, epochs=100,
                            shuffle=True, class_weight=class_weight_dict,
                            callbacks=callbacks)
        val_f1_scores = history.history['val_f1_score']
        best_f1 = max(val_f1_scores)
        best_epoch = val_f1_scores.index(best_f1)
        val_results = {
            'lr': lr,
            'best_epoch': best_epoch + 1,
            'f1': best_f1,
            'auc': history.history['val_auc'][best_epoch],
            'precision': history.history['val_precision'][best_epoch],
            'recall': history.history['val_recall'][best_epoch]
        }
        test_results = {}
        best_model = build_optimized_model(input_shape, lr)
        best_model.load_weights(checkpoint_path)
        for file in X_test_filenames:
            if not isinstance(file, str):
                continue
            match = re.search(r"chr\d+", file)
            chr_name = match.group() if match else f"file_{os.path.basename(file)}"
            try:
                X_test, _, y_test, _ = process_files_to_arrays([file])
                P_test = generate_P(X_test)
            except Exception as e:
                print(f"Warning: Failed to load {file}: {e}")
                continue
            y_pred_prob = best_model.predict(P_test, verbose=0).flatten()
            y_pred = (y_pred_prob > 0.5).astype(int)
            auc = tf.keras.metrics.AUC()(y_test, y_pred_prob).numpy()
            precision = tf.keras.metrics.Precision()(y_test, y_pred).numpy()
            recall = tf.keras.metrics.Recall()(y_test, y_pred).numpy()
            f1 = F1Score()(y_test, y_pred).numpy()
            test_results[chr_name] = {'lr': lr, 'auc': auc, 'precision': precision,
                                      'recall': recall, 'f1': f1}
        all_lr_results.append({'val': val_results, 'test': test_results})
    return all_lr_results



## 训练集、验证集、测试集划分

In [ ]:

# ======================== 数据准备与主程序 ========================
dir1 = '...path.../GM12878/model_data'
dir2 = '25_sub_matrix.txt'
chr_list = [f"chr{i}" for i in range(1, 23)]

def data(chr_list):
    file = []
    for chr in chr_list:
        if chr == "chr3" or chr == "chr18":
            continue
        f1 = os.path.join(dir1, f"{chr}_{dir2}")
        file.append(f1)
    X_train_filenames = file[:11]
    X_val_filenames = file[11:17]
    X_test_filenames = file[17:21]
    return X_test_filenames, X_train_filenames, X_val_filenames

X_test_filenames, X_train_filenames, X_val_filenames = data(chr_list)
print("Train files:", X_train_filenames)
print("Val files:", X_val_filenames)
print("Test files:", X_test_filenames)

Train files: ['/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr1_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr2_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr4_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr5_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr6_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr7_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr8_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr9_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr10_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr11_25_sub_matrix.txt', '/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr12_25_sub_matrix.txt']
Val files: ['/mnt/sde/zhangxiaoshuang/hygergraph/GM12878/model_data/chr13_25_sub_matrix.txt'

In [5]:
X_train, middle_row_indices_train, y_train, sample_counts_train = process_files_to_arrays(X_train_filenames)
X_val, middle_row_indices_val, y_val, sample_counts_val = process_files_to_arrays(X_val_filenames)
print_class_distribution(y_train, description="Train")
print_class_distribution(y_val, description="Val")

Train sample distribution:
Positive (1): 3299
Negative (0): 69969
Total: 73268
Ratio: 4.50% positive

Val sample distribution:
Positive (1): 1108
Negative (0): 18492
Total: 19600
Ratio: 5.65% positive



In [ ]:


def main():
    X_selected_train, mid_selected_train, y_selected_train = process_and_merge_chromosomes(
        X_train, middle_row_indices_train, y_train, sample_counts_train, select)
    X_selected_val, mid_selected_val, y_selected_val = process_and_merge_chromosomes(
        X_val, middle_row_indices_val, y_val, sample_counts_val, select)
    base_seeds = [42]
    repeats_per_seed = 4
    output_root = 'multi_seed_lr_experiment_results'
    os.makedirs(output_root, exist_ok=True)
    val_metrics_all = {'base_seed': [], 'repeat': [], 'lr': [],
                       'auc': [], 'precision': [], 'recall': [], 'f1': []}
    test_metrics_all = {}
    current_exp = 1
    total_experiments = len(base_seeds) * repeats_per_seed
    for base_seed in base_seeds:
        for repeat in range(repeats_per_seed):
            exp_seed = base_seed * 100 + repeat
            print(f"\n===== Experiment {current_exp}/{total_experiments} | Base seed: {base_seed} | Repeat: {repeat+1}/{repeats_per_seed} | Sub-seed: {exp_seed} =====")
            all_lr_results = run_experiment(
                seed=exp_seed,
                X_selected_train=X_selected_train,
                y_selected_train=y_selected_train,
                X_selected_val=X_selected_val,
                y_selected_val=y_selected_val,
                X_test_filenames=X_test_filenames
            )
            for lr_result in all_lr_results:
                lr = lr_result['val']['lr']
                val_res = lr_result['val']
                val_metrics_all['base_seed'].append(base_seed)
                val_metrics_all['repeat'].append(repeat+1)
                val_metrics_all['lr'].append(lr)
                val_metrics_all['auc'].append(val_res['auc'])
                val_metrics_all['precision'].append(val_res['precision'])
                val_metrics_all['recall'].append(val_res['recall'])
                val_metrics_all['f1'].append(val_res['f1'])
                test_res = lr_result['test']
                for chr_name, metrics in test_res.items():
                    if chr_name not in test_metrics_all:
                        test_metrics_all[chr_name] = {'base_seed': [], 'repeat': [], 'lr': [],
                                                      'auc': [], 'precision': [], 'recall': [], 'f1': []}
                    test_metrics_all[chr_name]['base_seed'].append(base_seed)
                    test_metrics_all[chr_name]['repeat'].append(repeat+1)
                    test_metrics_all[chr_name]['lr'].append(lr)
                    test_metrics_all[chr_name]['auc'].append(metrics['auc'])
                    test_metrics_all[chr_name]['precision'].append(metrics['precision'])
                    test_metrics_all[chr_name]['recall'].append(metrics['recall'])
                    test_metrics_all[chr_name]['f1'].append(metrics['f1'])
            current_exp += 1
    # 保存结果...
    val_df = pd.DataFrame(val_metrics_all)
    for lr in val_df['lr'].unique():
        lr_dir = os.path.join(output_root, f'lr_{lr}')
        os.makedirs(lr_dir, exist_ok=True)
        val_lr_df = val_df[val_df['lr'] == lr]
        val_lr_df.to_csv(os.path.join(lr_dir, 'validation_metrics_details.csv'), index=False)
        val_summary = pd.DataFrame({
            'metric': ['auc', 'precision', 'recall', 'f1'],
            'overall_mean±sd': [
                f"{val_lr_df['auc'].mean():.3f}±{val_lr_df['auc'].std(ddof=1):.3f}",
                f"{val_lr_df['precision'].mean():.3f}±{val_lr_df['precision'].std(ddof=1):.3f}",
                f"{val_lr_df['recall'].mean():.3f}±{val_lr_df['recall'].std(ddof=1):.3f}",
                f"{val_lr_df['f1'].mean():.3f}±{val_lr_df['f1'].std(ddof=1):.3f}"
            ]
        })
        val_summary.to_csv(os.path.join(lr_dir, 'validation_metrics_summary.csv'), index=False)
    for chr_name, metrics in test_metrics_all.items():
        test_df = pd.DataFrame(metrics)
        for lr in test_df['lr'].unique():
            lr_dir = os.path.join(output_root, f'lr_{lr}')
            os.makedirs(lr_dir, exist_ok=True)
            test_lr_df = test_df[test_df['lr'] == lr]
            test_lr_df.to_csv(os.path.join(lr_dir, f'test_{chr_name}_metrics_details.csv'), index=False)
            test_summary = pd.DataFrame({
                'metric': ['auc', 'precision', 'recall', 'f1'],
                'overall_mean±sd': [
                    f"{test_lr_df['auc'].mean():.3f}±{test_lr_df['auc'].std(ddof=1):.3f}",
                    f"{test_lr_df['precision'].mean():.3f}±{test_lr_df['precision'].std(ddof=1):.3f}",
                    f"{test_lr_df['recall'].mean():.3f}±{test_lr_df['recall'].std(ddof=1):.3f}",
                    f"{test_lr_df['f1'].mean():.3f}±{test_lr_df['f1'].std(ddof=1):.3f}"
                ]
            })
            test_summary.to_csv(os.path.join(lr_dir, f'test_{chr_name}_metrics_summary.csv'), index=False)
    print(f"\nAll results saved to {output_root}")

if __name__ == "__main__":
    main()

Chromosome 1 processed | pos: 480 | neg: 480 | total: 960
Chromosome 2 processed | pos: 400 | neg: 400 | total: 800
Chromosome 3 processed | pos: 309 | neg: 309 | total: 618
Chromosome 4 processed | pos: 270 | neg: 270 | total: 540
Chromosome 5 processed | pos: 293 | neg: 293 | total: 586
Chromosome 6 processed | pos: 286 | neg: 286 | total: 572
Chromosome 7 processed | pos: 248 | neg: 248 | total: 496
Chromosome 8 processed | pos: 243 | neg: 243 | total: 486
Chromosome 9 processed | pos: 258 | neg: 258 | total: 516
Chromosome 10 processed | pos: 265 | neg: 265 | total: 530
Chromosome 11 processed | pos: 247 | neg: 247 | total: 494

All merged | feature shape: (6598, 11, 50) | pos ratio: 3299/6598
Chromosome 1 processed | pos: 200 | neg: 200 | total: 400
Chromosome 2 processed | pos: 187 | neg: 187 | total: 374
Chromosome 3 processed | pos: 187 | neg: 187 | total: 374
Chromosome 4 processed | pos: 187 | neg: 187 | total: 374
Chromosome 5 processed | pos: 198 | neg: 198 | total: 396
Chr

2026-08-29 15:41:26.235902: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22052 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:14:00.0, compute capability: 8.9
2026-08-29 15:41:26.237483: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22052 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:25:00.0, compute capability: 8.9


2026-08-29 15:41:26.821627: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Epoch 1/100


2026-08-29 15:41:38.333063: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2026-08-29 15:41:38.419786: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-08-29 15:41:41.559181: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f7ecca9f190 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-08-29 15:41:41.559215: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-08-29 15:41:41.559221: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (1): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-08-29 15:41:41.591730: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787989301.787627 1697631 device_comp

...

Epoch 1/100
207/207 [==============================] - ETA: 0s - loss: 0.4025 - accuracy: 0.7434 - auc: 0.8133 - recall: 0.7472 - precision: 0.7416 - f1_score: 0.7444

/home/zhangxiaoshuang/anaconda3/envs/run_gpu/lib/python3.10/site-packages/keras/src/engine/training.py:2723: UserWarning: Metric F1Score implements a `reset_states()` method; rename it to `reset_state()` (without the final "s"). The name `reset_states()` has been deprecated to improve API consistency.
  m.reset_state()



Epoch 1: val_f1_score improved from -inf to 0.81291, saving model to checkpoints/seed_4203/lr_0.0003/best_model_seed_4203_lr_0.0003.weights.h5
207/207 [==============================] - 29s 70ms/step - loss: 0.4025 - accuracy: 0.7434 - auc: 0.8133 - recall: 0.7472 - precision: 0.7416 - f1_score: 0.7444 - val_loss: 0.3554 - val_accuracy: 0.8064 - val_auc: 0.8852 - val_recall: 0.8412 - val_precision: 0.7865 - val_f1_score: 0.8129
Epoch 2/100
206/207 [============================>.] - ETA: 0s - loss: 0.3387 - accuracy: 0.7819 - auc: 0.8490 - recall: 0.7645 - precision: 0.7919 - f1_score: 0.7779
Epoch 2: val_f1_score did not improve from 0.81291
207/207 [==============================] - 13s 63ms/step - loss: 0.3387 - accuracy: 0.7819 - auc: 0.8491 - recall: 0.7645 - precision: 0.7921 - f1_score: 0.7780 - val_loss: 0.3225 - val_accuracy: 0.7401 - val_auc: 0.8857 - val_recall: 0.5298 - val_precision: 0.9143 - val_f1_score: 0.6709
Epoch 3/100
207/207 [==============================] - ETA: 


All results saved to multi_seed_lr_experiment_results


: 